# 03 — Metric-learning embedder (Approach 2)

**Why this exists**: 02_train_siamese_v2 hit pair-F1 ≈ 0.98 but end-to-end
clustering F1 ≈ 0.0007 — catastrophic over-merging. Root cause: the classifier
head sees raw `cosine_sim(img,img)` and `cosine_sim(txt,txt)` as features and
gets a shortcut that's saturated for "different white shoes" confounders. The
hard-mined training pairs (sim ≥ 0.9) never taught the model to say "no" to
the 0.6–0.9 similarity band, which is where most KNN candidates land at
inference.

**This notebook removes the shortcut entirely.** No classifier head, no raw
cosine inputs. We train an embedder with **triplet margin loss** + semi-hard
mining over `items_train` labels. After training, every item gets encoded into
a 128-d unit vector. Clustering = cosine KNN in the learned space + connected
components. Same retrieval mechanism, same downstream pipeline, but the
embeddings are now directly optimised for the geometry the clustering step
needs.

> **Reminder, this is NOT an autoencoder.** No reconstruction loss. Same
> architecture as the v2 siamese **embedder** (img+text+price → 128d), trained
> with metric learning instead of pair classification.

**Plan:**
1. Load FashionCLIP image (512d) + multilingual text (512d).
2. Filter `items_train` to labels with ≥2 items, 80/20 label-disjoint split.
3. Train encoder with `pytorch-metric-learning` (`MPerClassSampler` m=4,
   `TripletMarginMiner` semi-hard, `TripletMarginLoss` margin=0.2).
4. Encode every item once → `phase2/embeddings/learned_embeddings.pt`.
5. End-to-end clustering on the same 30k val slice as 02 → submission +
   interactive viewer.


In [ ]:
import sys, os
sys.path.insert(0, '/Users/matouskovar/FIT/adm-sp/src')

REPO_ROOT          = '/Users/matouskovar/FIT/adm-sp'
DATA_DIR           = f'{REPO_ROOT}/data'
ARTIFACTS_DIR      = f'{REPO_ROOT}/artifacts'
PHASE2_DIR         = f'{REPO_ROOT}/phase2'

FASHION_EMB_PATH   = f'{PHASE2_DIR}/embeddings/fashion_clip_image.pt'
TEXT_EMB_PATH      = f'{ARTIFACTS_DIR}/embeddings/text_multilingual.pt'
PIPELINE_PATH      = f'{ARTIFACTS_DIR}/pipelines/preprocessing.pkl'
VOCAB_DIR          = f'{ARTIFACTS_DIR}/vocabularies'
VAL_PAIRS_PATH     = f'{ARTIFACTS_DIR}/pairs/val_pairs.csv'      # used to anchor the same val labels as 02

METRIC_V2_PATH     = f'{PHASE2_DIR}/models/metric_v2.pth'
LEARNED_EMB_PATH   = f'{PHASE2_DIR}/embeddings/learned_embeddings.pt'
VAL_SUBMISSION     = f'{PHASE2_DIR}/submissions/val_clusters_metric_v2.csv'

IMAGES_DIR         = '/Users/matouskovar/FIT/images'

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import pickle
import random
from collections import defaultdict, Counter
from tqdm.auto import tqdm

IMG_EMB_DIM  = 512
TEXT_EMB_DIM = 512
EMB_OUT_DIM  = 128

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# %pip install pytorch-metric-learning


## 1. Load embeddings, vocab, pipeline, items_train

In [ ]:
print('Loading FashionCLIP image embeddings...')
img_emb_dict = torch.load(FASHION_EMB_PATH, map_location='cpu', weights_only=False)
print(f'  {len(img_emb_dict):,} items, dim={next(iter(img_emb_dict.values())).shape[0]}')

print('Loading multilingual text embeddings...')
txt_emb_dict = torch.load(TEXT_EMB_PATH, map_location='cpu', weights_only=False)
print(f'  {len(txt_emb_dict):,} items, dim={next(iter(txt_emb_dict.values())).shape[0]}')

from GlamiDatasetVocabulary import GlamiVocabularyManager
vocab_manager = GlamiVocabularyManager.load_from_dir(VOCAB_DIR)
pipeline = pickle.load(open(PIPELINE_PATH, 'rb'))
print(f'Vocab: geo={len(vocab_manager.geo_dict)}  color={len(vocab_manager.color_dict)}  dept={len(vocab_manager.dept_dict)}')


In [ ]:
df_train = pd.read_csv(f'{DATA_DIR}/items_train.csv')
print(f'items_train: {len(df_train):,}  |  unique labels: {df_train.label.nunique():,}')

X = df_train.drop(columns=['label'])
df_train_t = pipeline.transform(X)
df_train_t['label']  = df_train['label'].values
df_train_t['itemId'] = df_train['itemId'].values

# Keep only items that have BOTH embeddings (defensive)
have_emb = set(img_emb_dict.keys()) & set(txt_emb_dict.keys())
df_train_t = df_train_t[df_train_t['itemId'].astype(str).isin(have_emb)].reset_index(drop=True)
print(f'After embedding filter: {len(df_train_t):,}')

# Filter to labels with >= 2 items (triplet anchors need positives)
label_counts = df_train_t['label'].value_counts()
multi_labels = label_counts[label_counts >= 2].index
df_multi     = df_train_t[df_train_t['label'].isin(multi_labels)].reset_index(drop=True)
print(f'Multi-instance labels: {len(multi_labels):,}  ({len(df_multi):,} items)')


## 2. Label-disjoint 80/20 split

Same construction as the baseline: identify val labels from the existing
`val_pairs.csv` so the held-out evaluation matches what 02 used. Items from
val labels never appear in training → honest generalisation signal.


In [ ]:
val_pairs = pd.read_csv(VAL_PAIRS_PATH)
pos_val   = val_pairs[val_pairs['is_duplicate'] == 1.0]
val_pos_items = set(pos_val['item_id_1'].astype(str)) | set(pos_val['item_id_2'].astype(str))

train_label_lookup = dict(zip(df_train['itemId'].astype(str), df_train['label']))
val_label_set = {train_label_lookup[i] for i in val_pos_items if i in train_label_lookup}

train_label_set = set(multi_labels) - val_label_set

df_split_train = df_multi[df_multi['label'].isin(train_label_set)].reset_index(drop=True)
df_split_val   = df_multi[df_multi['label'].isin(val_label_set)].reset_index(drop=True)

print(f'Training labels: {len(train_label_set):,}  items: {len(df_split_train):,}')
print(f'Val labels    : {len(val_label_set):,}    items: {len(df_split_val):,}')
print(f'Item overlap  : {len(set(df_split_train.itemId) & set(df_split_val.itemId))}  (must be 0)')


## 3. Dataset (returns features + integer label)

In [ ]:
class MetricDataset(Dataset):
    """Per-item: feature dict + integer label (consecutive ids needed by samplers)."""
    def __init__(self, df, vocab, img_dict, txt_dict, label_to_idx=None):
        self.df    = df.reset_index(drop=True)
        self.vocab = vocab
        self.img_dict = img_dict
        self.txt_dict = txt_dict
        if label_to_idx is None:
            label_to_idx = {lbl: i for i, lbl in enumerate(sorted(self.df['label'].unique()))}
        self.label_to_idx = label_to_idx
        self.labels_int = self.df['label'].map(label_to_idx).values.astype(np.int64)

    def __len__(self):
        return len(self.df)

    def _multi_hot(self, ids_string, vocab_dict):
        t = torch.zeros(len(vocab_dict), dtype=torch.float32)
        if pd.isna(ids_string) or str(ids_string).strip() == '':
            return t
        for raw_id in str(ids_string).split(','):
            raw_id = raw_id.strip()
            if raw_id in vocab_dict:
                t[vocab_dict[raw_id]] = 1.0
        return t

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        iid = str(row['itemId'])
        img = self.img_dict.get(iid, torch.zeros(IMG_EMB_DIM))
        txt = self.txt_dict.get(iid, torch.zeros(TEXT_EMB_DIM))
        if img.device.type != 'cpu': img = img.cpu()
        if txt.device.type != 'cpu': txt = txt.cpu()
        img = F.normalize(img.float(), dim=0)
        txt = F.normalize(txt.float(), dim=0)
        feats = {
            'image_embedding': img,
            'text_embedding':  txt,
            'price':           torch.tensor([row['price_scaled']], dtype=torch.float32),
        }
        return feats, int(self.labels_int[idx])

def collate(batch):
    feats, lbls = zip(*batch)
    out = {k: torch.stack([f[k] for f in feats]) for k in feats[0]}
    return out, torch.tensor(lbls, dtype=torch.long)

# Build a SHARED label-to-int map across train + val (so val label ints are stable)
all_labels = sorted(set(df_split_train['label'].unique()) | set(df_split_val['label'].unique()))
label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}

ds_train = MetricDataset(df_split_train, vocab_manager, img_emb_dict, txt_emb_dict, label_to_idx)
ds_val   = MetricDataset(df_split_val,   vocab_manager, img_emb_dict, txt_emb_dict, label_to_idx)
print(f'Train ds: {len(ds_train):,}  |  Val ds: {len(ds_val):,}')


## 4. Encoder (same SiameseEmbedder, L2-normalized output)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, img_dim=IMG_EMB_DIM, txt_dim=TEXT_EMB_DIM, hidden=256, out_dim=EMB_OUT_DIM):
        super().__init__()
        self.image_proj = nn.Sequential(
            nn.Linear(img_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2))
        self.text_proj = nn.Sequential(
            nn.Linear(txt_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2))
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2 + 1, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, out_dim))

    def forward(self, feats):
        img = self.image_proj(feats['image_embedding'])
        txt = self.text_proj(feats['text_embedding'])
        x   = torch.cat([img, txt, feats['price']], dim=1)
        emb = self.fusion(x)
        return F.normalize(emb, p=2, dim=1)   # unit-norm: cosine = dot product

encoder = Encoder().to(device)
print(f'Trainable params: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}')


## 5. Sampler + loss + miner (pytorch-metric-learning)

In [ ]:
from pytorch_metric_learning import losses, miners, samplers

M_PER_CLASS = 4
N_CLASSES_PER_BATCH = 64
BATCH_SIZE = M_PER_CLASS * N_CLASSES_PER_BATCH

# Reweight: items belonging to the same label are guaranteed in the same batch
sampler_train = samplers.MPerClassSampler(
    labels=ds_train.labels_int,
    m=M_PER_CLASS,
    batch_size=BATCH_SIZE,
    length_before_new_iter=len(ds_train),
)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler_train,
                          num_workers=0, collate_fn=collate, drop_last=True)
print(f'Train batches per epoch: {len(train_loader):,}  (BS={BATCH_SIZE})')

# Triplet margin loss with semi-hard mining
miner   = miners.TripletMarginMiner(margin=0.2, type_of_triplets='semihard')
loss_fn = losses.TripletMarginLoss(margin=0.2)


## 6. Validation: same-label vs different-label cosine separation

Cheap label-free probe of how well the embedder separates products. Mean
cosine of same-label pairs minus mean cosine of different-label pairs. Useful
to track per epoch — should rise from ~0 toward ~0.5+.


In [ ]:
@torch.no_grad()
def encode_all(encoder, ds, batch=1024):
    encoder.eval()
    out = []
    loader = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=0, collate_fn=collate)
    for feats, _ in loader:
        feats = {k: v.to(device) for k, v in feats.items()}
        out.append(encoder(feats).cpu())
    return torch.cat(out, dim=0)

@torch.no_grad()
def val_separation(encoder, ds_val, n_pairs=2000, seed=0):
    embs = encode_all(encoder, ds_val)
    labels = ds_val.labels_int

    rng = np.random.default_rng(seed)
    by_label = defaultdict(list)
    for i, l in enumerate(labels):
        by_label[int(l)].append(i)
    multi = [v for v in by_label.values() if len(v) >= 2]
    rng.shuffle(multi)

    same, diff = [], []
    for v in multi:
        i, j = rng.choice(v, 2, replace=False)
        same.append(float(embs[i] @ embs[j]))
        if len(same) >= n_pairs: break
    while len(diff) < n_pairs:
        i, j = rng.integers(0, len(embs), size=2)
        if labels[i] != labels[j]:
            diff.append(float(embs[i] @ embs[j]))

    return np.mean(same), np.mean(diff), np.mean(same) - np.mean(diff)

s, d, gap = val_separation(encoder, ds_val)
print(f'BEFORE training:  same={s:.3f}  diff={d:.3f}  gap={gap:+.3f}')


## 7. Training loop

In [ ]:
NUM_EPOCHS = 12
LR         = 1e-3

opt   = torch.optim.AdamW(encoder.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)

os.makedirs(os.path.dirname(METRIC_V2_PATH), exist_ok=True)

history = {'loss': [], 'gap': []}
best_gap = -1.0

for epoch in range(NUM_EPOCHS):
    encoder.train()
    epoch_loss, n_triplets = 0.0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for feats, lbls in pbar:
        feats = {k: v.to(device) for k, v in feats.items()}
        lbls  = lbls.to(device)

        emb = encoder(feats)
        triplets = miner(emb, lbls)
        loss = loss_fn(emb, lbls, triplets)

        opt.zero_grad()
        loss.backward()
        opt.step()

        epoch_loss  += float(loss) * len(emb)
        n_triplets  += int(miner.num_triplets)
        pbar.set_postfix(loss=f'{float(loss):.4f}', triplets=miner.num_triplets)

    sched.step()
    avg_loss = epoch_loss / max(len(ds_train), 1)
    s, d, gap = val_separation(encoder, ds_val)
    history['loss'].append(avg_loss); history['gap'].append(gap)

    print(f'  loss={avg_loss:.4f}  triplets/epoch={n_triplets:,}  same={s:.3f}  diff={d:.3f}  gap={gap:+.3f}')

    if gap > best_gap:
        best_gap = gap
        torch.save(encoder.state_dict(), METRIC_V2_PATH)
        print(f'  Saved best to {METRIC_V2_PATH}  (gap={gap:+.3f})')

print(f'\nBest val gap: {best_gap:+.3f}')


In [ ]:
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3))
ax1.plot(history['loss']); ax1.set_title('Triplet loss'); ax1.set_xlabel('epoch')
ax2.plot(history['gap']);  ax2.set_title('Same/diff cosine gap'); ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()


## 8. Encode every item once → `learned_embeddings.pt`

Unlike the v2 siamese, inference doesn't need a per-edge model call. Encode all
items once, then everything downstream is just cosine on a 128-d matrix.


In [ ]:
encoder.load_state_dict(torch.load(METRIC_V2_PATH, map_location=device, weights_only=True))
encoder.eval()

# Build a dataset spanning every item we have an embedding for (train + phase1 [+ phase2])
df_all = pd.read_csv(f'{DATA_DIR}/items_train.csv')
for extra in [f'{DATA_DIR}/items_phase_1.csv', f'{DATA_DIR}/items_phase_2.csv']:
    if os.path.exists(extra):
        df_all = pd.concat([df_all, pd.read_csv(extra)], ignore_index=True)
df_all = df_all.drop_duplicates(subset='itemId').reset_index(drop=True)

# Run through pipeline (drop label first if present)
X_all = df_all.drop(columns=['label'], errors='ignore')
df_all_t = pipeline.transform(X_all)
df_all_t['itemId'] = df_all['itemId'].values
# Add a dummy label column (label_to_idx defaults won't matter — we only use feats)
df_all_t['label'] = 0
df_all_t = df_all_t[df_all_t['itemId'].astype(str).isin(have_emb)].reset_index(drop=True)
print(f'Items to encode: {len(df_all_t):,}')

ds_all = MetricDataset(df_all_t, vocab_manager, img_emb_dict, txt_emb_dict,
                       label_to_idx={0: 0})

@torch.no_grad()
def encode_to_dict(encoder, ds, batch=1024):
    out = {}
    loader = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=0, collate_fn=collate)
    idx = 0
    for feats, _ in tqdm(loader, desc='Encoding all items'):
        feats = {k: v.to(device) for k, v in feats.items()}
        emb = encoder(feats).cpu()
        for j in range(emb.shape[0]):
            iid = str(ds.df.iloc[idx + j]['itemId'])
            out[iid] = emb[j]
        idx += emb.shape[0]
    return out

learned = encode_to_dict(encoder, ds_all)
os.makedirs(os.path.dirname(LEARNED_EMB_PATH), exist_ok=True)
torch.save(learned, LEARNED_EMB_PATH)
print(f'Saved {len(learned):,} learned embeddings → {LEARNED_EMB_PATH}')


## 9. End-to-end clustering on the 30k val slice

Same val construction as 02 so F1 numbers are directly comparable. **No
siamese rescoring** — clustering is just cosine in the learned space then
connected components.


In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

VAL_TOTAL = 30_000

df_dup_items   = df_train[df_train['label'].isin(val_label_set)]
n_dup          = min(len(df_dup_items), VAL_TOTAL // 2)
n_noise        = VAL_TOTAL - n_dup
df_dup_sample  = df_dup_items.sample(n=n_dup, random_state=42)
df_noise       = df_train[~df_train['label'].isin(val_label_set)].sample(n=n_noise, random_state=42)
df_val_all     = pd.concat([df_dup_sample, df_noise], ignore_index=True)
df_val_all['itemId'] = df_val_all['itemId'].astype(str)
df_val_all = df_val_all[df_val_all['itemId'].isin(learned)].reset_index(drop=True)

val_ids    = df_val_all['itemId'].tolist()
val_labels = dict(zip(df_val_all['itemId'], df_val_all['label']))
N = len(val_ids)
print(f'Val items: {N:,}  (dup-label: {n_dup:,}  noise: {n_noise:,})')

# Stack learned embeddings into a matrix (already L2-normed)
val_mat = torch.stack([learned[iid] for iid in val_ids]).to(device)
val_mat = F.normalize(val_mat, p=2, dim=1)
print(f'Val matrix: {val_mat.shape}')


In [ ]:
# Cosine KNN in blocks; collect all neighbours above threshold (since cos==score)
KNN_THRESHOLD = 0.50   # learned space is more discriminative; we can be more permissive
TOP_K         = 40
BLOCK         = 512

all_i, all_j, all_s = [], [], []
for start in tqdm(range(0, N, BLOCK), desc='KNN'):
    end = min(start + BLOCK, N)
    block = val_mat[start:end]
    sims  = torch.mm(block, val_mat.T)
    for bi in range(end - start):
        sims[bi, start + bi] = -2.0
    vals_k, idx_k = torch.topk(sims, k=TOP_K, dim=1)
    for bi in range(end - start):
        gi = start + bi
        for ki in range(TOP_K):
            gj = idx_k[bi, ki].item()
            sim = vals_k[bi, ki].item()
            if sim >= KNN_THRESHOLD and gj > gi:
                all_i.append(gi); all_j.append(gj); all_s.append(sim)

all_i = np.array(all_i, dtype=np.int32)
all_j = np.array(all_j, dtype=np.int32)
all_s = np.array(all_s, dtype=np.float32)
print(f'Candidate edges (cos >= {KNN_THRESHOLD}): {len(all_i):,}')
print(f'Score range: [{all_s.min():.3f}, {all_s.max():.3f}]')


In [ ]:
def evaluate_clustering(id_to_cluster, id_to_label):
    common = list(set(id_to_cluster) & set(id_to_label))
    true_arr = np.array([id_to_label[i]   for i in common])
    pred_arr = np.array([id_to_cluster[i] for i in common])
    def c2(arr):
        _, c = np.unique(arr, return_counts=True)
        return int((c * (c - 1) // 2).sum()), int(c.max() if len(c) else 0)
    total_true, _      = c2(true_arr)
    total_pred, mx     = c2(pred_arr)
    composite          = np.array([f'{t}|{p}' for t, p in zip(true_arr, pred_arr)])
    tp, _              = c2(composite)
    fp = total_pred - tp; fn = total_true - tp
    p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
    f = (2 * p * r / max(p + r, 1e-9))
    return p, r, f, mx

def edges_to_clusters(rows, cols, n, ids):
    if not len(rows):
        return {iid: i for i, iid in enumerate(ids)}
    data = np.ones(2 * len(rows), dtype=np.int8)
    r = np.concatenate([rows, cols]); c = np.concatenate([cols, rows])
    _, lbl = connected_components(csr_matrix((data, (r, c)), shape=(n, n)), directed=False)
    return {ids[i]: int(lbl[i]) for i in range(n)}

print('threshold |  edges    P       R       F1     max_cluster')
print('-' * 60)
best = (0.0, 0.0, None)
for thr in [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.92, 0.95]:
    mask = all_s >= thr
    er, ec = all_i[mask], all_j[mask]
    id2c   = edges_to_clusters(er, ec, N, val_ids)
    p, r, f, mx = evaluate_clustering(id2c, val_labels)
    print(f'  {thr:.2f}    |  {mask.sum():>6,}   {p:.3f}   {r:.3f}   {f:.3f}   {mx:>5,}')
    if f > best[1]:
        best = (thr, f, id2c)

BEST_THR, BEST_F1, BEST_ID2C = best
print(f'\nBest threshold {BEST_THR}  -> end-to-end pairwise F1 = {BEST_F1:.4f}  (siamese baseline ~0.85, v2 siamese ~0.0007)')


In [ ]:
# Cluster size distribution at the best threshold
sizes = Counter(BEST_ID2C.values())
size_arr = sorted(sizes.values(), reverse=True)
print(f'Underlying connected components: {len(size_arr):,}')
print(f'  Singletons       : {sum(1 for s in size_arr if s == 1):,}')
print(f'  Multi-item       : {sum(1 for s in size_arr if s > 1):,}')
print(f'  Largest cluster  : {size_arr[0]:,}')
print(f'  Top 10 sizes     : {size_arr[:10]}')

total_pred = sum(s * (s - 1) // 2 for s in size_arr)
top1 = size_arr[0] * (size_arr[0] - 1) // 2
print(f'\nTotal predicted pairs : {total_pred:,}')
print(f'  from top-1 cluster  : {top1:,}  ({100*top1/max(total_pred,1):.1f}%)  (high % = mega-cluster bug)')


In [ ]:
# Write the best clustering as a submission file (≤100 items per line)
os.makedirs(os.path.dirname(VAL_SUBMISSION), exist_ok=True)

cluster_to_items = defaultdict(list)
for iid, cid in BEST_ID2C.items():
    cluster_to_items[cid].append(iid)

MAX_PER_LINE = 100
lines = []
for members in cluster_to_items.values():
    for s in range(0, len(members), MAX_PER_LINE):
        lines.append(','.join(members[s:s + MAX_PER_LINE]))

with open(VAL_SUBMISSION, 'w') as f:
    f.write('\n'.join(lines))

multi_n = sum(1 for m in cluster_to_items.values() if len(m) > 1)
print(f'Wrote {VAL_SUBMISSION}')
print(f'  Lines: {len(lines):,}  |  Multi-item clusters (pre-chunk): {multi_n:,}  |  Singletons: {sum(1 for m in cluster_to_items.values() if len(m) == 1):,}')


## 10. Interactive sanity check

Press **Enter** for a random multi-item cluster from the submission file:
- 🟦 anchor (first item in the line)
- 🟢 same true label (correct)
- 🔴 different label (false positive merge)

Type `q` + Enter to stop.


In [ ]:
import matplotlib.patches as mpatches
from PIL import Image as _PILImage

df_train['itemId'] = df_train['itemId'].astype(str)
val_meta = df_train[df_train['itemId'].isin(set(val_ids))].set_index('itemId')[['title','geo','label']].to_dict('index')

with open(VAL_SUBMISSION) as f:
    submission_groups = [line.strip().split(',') for line in f if line.strip()]
multi_groups = [g for g in submission_groups if len(g) > 1]
print(f'Multi-item groups in submission: {len(multi_groups):,}')

def _img(iid):
    try:
        return _PILImage.open(f'{IMAGES_DIR}/{iid}.jpg').convert('RGB')
    except Exception:
        return None

def show_random_cluster():
    if not multi_groups:
        print('No multi-item groups.'); return
    members = random.choice(multi_groups)
    anchor       = members[0]
    anchor_label = val_meta.get(anchor, {}).get('label')
    member_labels = [val_meta.get(m, {}).get('label') for m in members]
    correct = (len(set(member_labels)) == 1)

    n = len(members)
    fig, axes = plt.subplots(1, n, figsize=(min(4 * n, 22), 5.5))
    if n == 1: axes = [axes]
    for ax, iid, lbl in zip(axes, members, member_labels):
        im = _img(iid)
        if im is None:
            ax.text(0.5, 0.5, 'no image', ha='center', va='center', transform=ax.transAxes)
            ax.set_facecolor('#eeeeee')
        else:
            ax.imshow(im)
        if iid == anchor:        color, lw = '#1565C0', 4
        elif lbl == anchor_label: color, lw = '#2e7d32', 3
        else:                    color, lw = '#c62828', 3
        for sp in ax.spines.values():
            sp.set_edgecolor(color); sp.set_linewidth(lw)
        meta = val_meta.get(iid, {})
        ax.set_title(f'[{iid}]\n{str(meta.get("title",""))[:50]}\ngeo={meta.get("geo","?")}  label={lbl}',
                     fontsize=8, pad=4)
        ax.set_xticks([]); ax.set_yticks([])

    flag = 'CORRECT' if correct else 'MIXED (over-merge)'
    fig.suptitle(f'Group of {n}  [{flag}]   anchor_label={anchor_label}', fontsize=11, fontweight='bold')
    legend = [mpatches.Patch(color='#1565C0', label='Anchor'),
              mpatches.Patch(color='#2e7d32', label='Same true label'),
              mpatches.Patch(color='#c62828', label='Different label (FP)')]
    fig.legend(handles=legend, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(); plt.show()

print('Press Enter to draw. Type "q" + Enter to stop.\n')
while True:
    cmd = input('> ').strip().lower()
    if cmd == 'q': break
    show_random_cluster()
